# CRISP-DM Fase 1: Business Understanding (Comprensión del Negocio)
## Dataset: ICA Censo Pecuario e Inventarios Municipales (ICA PowerBI)

**Ecosistema**: AgroStats Intelligence Platform (`AgroStatsApp`)  
**Metodología**: CRISP-DM (Fase 1: Business Understanding) | **Fase PDCO**: PLAN  
**Estándares Aplicados**: SWEBOK Cap. 1 (Requerimientos de Software), DAMA-DMBOK 2 (Metadata & Governance), ISO/IEC 25010.

---

### 1. Contexto Estratégico y Problema de Negocio
El inventario pecuario es la variable fundamental de stock de capital animal. Su desglose espacial a nivel municipal permite aplicar índices de concentración geográfica (HHI, CR5) y modelos de densidad ganadera/porcina.

**Objetivo de Negocio**:
> Caracterizar el inventario pecuario nacional por especie y municipio DIVIPOLA para estimar densidad animal, capacidad productiva y concentraciones territoriales.

### 2. Entidad del Dominio y Alcance
* **Entidad Analítica Principal**: `Población Pecuaria Municipal (Bovino, Porcino, Avícola, Ovino-Caprino, Equino, Bufalino)`
* **Custodio Oficial de los Datos**: Instituto Colombiano Agropecuario (ICA) - Dirección Tecnológica / Subdirección de Sanidad Animal
* **Granularidad Espacial**: `Municipio (Código DIVIPOLA 5 dígitos DDMMM)`
* **Granularidad Temporal**: `Anual (Ciclos de Vacunación RUV / Censo Pecuario)`

### 3. Preguntas de Negocio Mapeadas (`docs/1-Bateria_preguntas.md`):
* **C1: Concentración Pecuaria (HHI)**
* **C2: Razón de Concentración Municipal (CR5)**
* **I1: Oferta Pecuaria vs. Abastecimiento de Carne**

### 4. Indicadores Clave de Desempeño (KPIs):
* `Inventario total de cabezas`
* `Índice HHI de concentración pecuaria`
* `Densidad pecuaria municipal por especie`

---

### 📋 FICHA TÉCNICA DEL DATASET (DAMA-DMBOK 2)

#### 📦 A. QUÉ HABÍA EN EL DATASET ORIGINAL (Diagnóstico Raw)
> **Estado de Origen**: Dashboard incrustado en PowerBI (app.powerbi.com/view) sin descarga directa estática CSV; datos fragmentados por especie (bovino, porcino, avícola, etc.) y municipios con texto libre.

#### 🔧 B. QUÉ SE NECESITA PARA CONSTRUIR LOS INDICADORES (Transformaciones y Fórmulas)
> **Requisitos y Formulación**: Extracción multi-estrategia vía ICAPowerBIExtractor (PowerBI Query API / XLSX Censo), sanitización del código municipal DIVIPOLA de 5 dígitos (DDMMM), masa pecuaria por especie, índice HHI y densidad por hectárea.

---


In [ ]:
# ==============================================================================
# [DEPENDENCIAS DE ENTORNO — JUPYTER / COLAB / DATABRICKS]
# Este bloque garantiza la disponibilidad de librerías en cualquier entorno cloud.
# ==============================================================================
%pip install -q pandas numpy requests python-dotenv openpyxl pypdf pyarrow pyreadstat matplotlib seaborn scipy statsmodels scikit-learn duckdb pydantic missingno


In [ ]:
# ==============================================================================
# [CONFIGURACIÓN DEL KERNEL Y RESOLUCIÓN ROBUSTA DE RUTAS DEL PROYECTO (PEP 8)]
# Compatible con Local Jupyter, VSCode, Colab, Databricks Git Folders & Repos
# ==============================================================================
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Búsqueda dinámica ascendente de la raíz del proyecto (donde reside src/ y metadata.json)
CURRENT_DIR = Path(".").resolve()
APP_ROOT = None

for candidate in [CURRENT_DIR, CURRENT_DIR.parent, CURRENT_DIR.parent.parent, CURRENT_DIR.parent.parent.parent]:
    if (candidate / "src" / "database").exists() or (candidate / "metadata.json").exists():
        APP_ROOT = candidate
        break

if APP_ROOT is None:
    candidate = CURRENT_DIR
    while candidate.parent != candidate:
        if (candidate / "src" / "database").exists():
            APP_ROOT = candidate
            break
        candidate = candidate.parent

if APP_ROOT is None:
    APP_ROOT = CURRENT_DIR

if str(APP_ROOT) not in sys.path:
    sys.path.insert(0, str(APP_ROOT))

import pandas as pd
import numpy as np

print(f"✅ Entorno e importaciones inicializadas exitosamente.")
print(f"📍 Raíz del proyecto detectada: {APP_ROOT}")


### 5. Fundamentación del Contrato de Datos (Data Contract)
Bajo los lineamientos de **DAMA-DMBOK 2**, todo pipeline de datos analíticos debe iniciar con la verificación de un **Contrato de Datos**. 
El contrato define formalmente:
1. El custodio del dato y la periodicidad de actualización.
2. La definición operativa de las variables críticas y sus tipos de datos esperados.
3. Las tolerancias de calidad (porcentaje máximo de nulos y umbrales de detección de valores anómalos).


In [ ]:
# ==============================================================================
# [FASE 1: VERIFICACIÓN DEL CONTRATO DE NEGOCIO Y ESPECIFICACIÓN TÉCNICA]
# ==============================================================================
import yaml

config_path = APP_ROOT / "config" / "datasets_config.yaml"
if config_path.exists():
    with open(config_path, "r", encoding="utf-8") as f:
        config_data = yaml.safe_load(f)
    print(f"📄 Archivo de configuración cargado: {config_path.name}")
    dataset_cfg = config_data.get("datasets", {}).get("10_ica_inventario_pecuario", {})
    print(f"📋 Especificación del Dataset en YAML: {dataset_cfg}")
else:
    print("ℹ️ Configuración YAML por defecto activa.")

print("\n--- DEFINICIÓN FORMAL DEL CONTRATO ---")
print("• Entidad:", "Población Pecuaria Municipal (Bovino, Porcino, Avícola, Ovino-Caprino, Equino, Bufalino)")
print("• Custodio:", "Instituto Colombiano Agropecuario (ICA) - Dirección Tecnológica / Subdirección de Sanidad Animal")
print("• Granularidad Espacial:", "Municipio (Código DIVIPOLA 5 dígitos DDMMM)")
print("• Granularidad Temporal:", "Anual (Ciclos de Vacunación RUV / Censo Pecuario)")
print("• KPIs de Negocio:", ['Inventario total de cabezas', 'Índice HHI de concentración pecuaria', 'Densidad pecuaria municipal por especie'])


### 6. Verificación de Disponibilidad en el Data Lakehouse
El Lakehouse implementa la arquitectura Medallion:
* **Bronze**: Datos crudos inmutables preservando el formato original del custodio.
* **Silver**: Datos limpios, sanitizados y estructurados en formato columnar Parquet comprimido con Snappy.
* **Gold**: Base de datos relacional SQLite (`agrostats_lakehouse.db`) con esquemas conformados e índices para consultas analíticas de alta velocidad.


In [ ]:
# ==============================================================================
# [FASE 1: INSPECCIÓN DE DISPONIBILIDAD EN LAS CAPAS DEL LAKEHOUSE]
# ==============================================================================
from src.database.db_manager import DatabaseManager

db_path = APP_ROOT / "data" / "processed" / "agrostats_lakehouse.db"
parquet_path = APP_ROOT / "data/processed/ica_inventario_pecuario.parquet"

print("🔍 Verificación de Almacenamiento en Data Lakehouse:")
print(f"• Capa Silver (Parquet): {parquet_path.name} -> Existe: {parquet_path.exists()}")
print(f"• Capa Gold (SQLite DB): {db_path.name} -> Existe: {db_path.exists()}")

if parquet_path.exists():
    df_sample = pd.read_parquet(parquet_path)
    print(f"\n✅ Contrato de datos preliminar verificado exitosamente:")
    print(f"• Total de registros curados: {len(df_sample):,} filas")
    print(f"• Columnas disponibles: {list(df_sample.columns)}")


### 7. Síntesis y Transición a la Fase 2 (Data Understanding)
* **Entregable de la Fase 1**: Especificación formal del problema de negocio, KPIs concertados y contrato de datos validado.
* **Siguiente Paso**: Proceder al notebook `02_ingestion.ipynb` para ejecutar la extracción inmutable y auditoría criptográfica SHA-256.
